# 03 — CNN: Squeeze Maximum from EfficientNet-B0

## Setup

In [9]:
import os, glob, random, warnings, time, gc, copy
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from dataclasses import dataclass, field
from typing import List
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import torchaudio
import timm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}, GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

@dataclass
class Config:
    data_root: str = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
    output_dir: str = '/kaggle/working'
    sr: int = 16000
    duration: float = 10.24
    n_mels: int = 128
    n_fft: int = 1024
    hop_length: int = 320
    fmin: int = 20
    fmax: int = 8000
    genres: List[str] = field(default_factory=lambda: sorted([
        'blues','classical','country','disco','hiphop',
        'jazz','metal','pop','reggae','rock']))
    stem_types: List[str] = field(default_factory=lambda: ['drums','vocals','bass'])
    entity: str = "23f3003225-indian-institute-of-technology-madras"
    project: str = "23f3003225-dl-genai-project"
    @property
    def stems_dir(self): return os.path.join(self.data_root, 'genres_stems')
    @property
    def noise_dir(self): return os.path.join(self.data_root, 'ESC-50-master', 'audio')
    @property
    def test_dir(self): return os.path.join(self.data_root, 'mashups')
    @property
    def test_csv(self): return os.path.join(self.data_root, 'test.csv')

CFG = Config()
TARGET_LEN = int(CFG.sr * CFG.duration)  # 163840
GENRE2IDX = {g: i for i, g in enumerate(CFG.genres)}
IDX2GENRE = {i: g for g, i in GENRE2IDX.items()}
STEM_WEIGHTS = {'drums': 0.45, 'vocals': 0.35, 'bass': 0.20}

os.makedirs(CFG.output_dir, exist_ok=True)
print(f"SR={CFG.sr}, target_len={TARGET_LEN}, {len(CFG.genres)} genres")

Device: cpu, GPU: N/A
SR=16000, target_len=163840, 10 genres


## WandB

In [10]:
os.system('pip install wandb --upgrade --no-cache-dir -q')
import wandb
wandb.login(key="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")

run = wandb.init(
    entity=CFG.entity, project=CFG.project, name="03_cnn_efficientnet",
    config={"phase": "cnn", "sr": CFG.sr, "duration": CFG.duration, "n_mels": CFG.n_mels,
            "backbone": "efficientnet_b0", "pooling": "gem"},
    tags=["cnn", "efficientnet", "phase3"], job_type="train",
)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


## Data Index

In [11]:
data_index = {g: [] for g in CFG.genres}
for genre in CFG.genres:
    gp = os.path.join(CFG.stems_dir, genre)
    songs = sorted(s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s)))
    for song in songs:
        song_dir = os.path.join(gp, song)
        avail = [st for st in CFG.stem_types if os.path.exists(os.path.join(song_dir, f"{st}.wav"))]
        if avail:
            data_index[genre].append({'dir': song_dir, 'stems': avail})
for g in CFG.genres: print(f"  {g}: {len(data_index[g])} songs")

# Flat arrays for K-Fold
all_songs, all_labels = [], []
for genre in CFG.genres:
    for s in data_index[genre]:
        all_songs.append(s)
        all_labels.append(GENRE2IDX[genre])
all_labels = np.array(all_labels)

noise_files = sorted(glob.glob(os.path.join(CFG.noise_dir, "*.wav")))
print(f"\nTotal: {len(all_songs)} songs, {len(noise_files)} noise clips")

  blues: 100 songs
  classical: 100 songs
  country: 100 songs
  disco: 100 songs
  hiphop: 100 songs
  jazz: 100 songs
  metal: 100 songs
  pop: 100 songs
  reggae: 100 songs
  rock: 100 songs

Total: 1000 songs, 2000 noise clips


## Audio Utils

In [12]:
def load_wav(path, sr=CFG.sr, target_len=TARGET_LEN):
    try:
        wav, orig_sr = torchaudio.load(path)
        if wav.shape[0] > 1: wav = wav.mean(dim=0, keepdim=True)
        if orig_sr != sr: wav = torchaudio.functional.resample(wav, orig_sr, sr)
        wav = wav.squeeze(0)
        if len(wav) < target_len: wav = F.pad(wav, (0, target_len - len(wav)))
        elif len(wav) > target_len:
            start = random.randint(0, len(wav) - target_len)
            wav = wav[start:start + target_len]
        return wav
    except:
        return torch.zeros(target_len)

print("Audio utils ready.")

Audio utils ready.


## Datasets
**MashupDataset**: on-the-fly synthetic mashups with noise, time shift, overdrive, pitch shift, SpecAugment, Mixup.
**ValDataset**: clean stem mix (no augmentation).
**TestDataset**: loads mashup wav files directly.

In [13]:
class MashupDataset(Dataset):
    def __init__(self, song_list, label_list, noise_files, samples_per_genre=800,
                 augment=True, use_specaug=False, mixup_alpha=0.0):
        self.noise_files = noise_files
        self.augment = augment
        self.use_specaug = use_specaug
        self.mixup_alpha = mixup_alpha
        # Group by genre
        self.genre_songs = {g: [] for g in CFG.genres}
        for song, label in zip(song_list, label_list):
            self.genre_songs[IDX2GENRE[label]].append(song)
        self.samples = []
        for genre in CFG.genres:
            if self.genre_songs[genre]:
                for _ in range(samples_per_genre):
                    self.samples.append(GENRE2IDX[genre])

    def __len__(self): return len(self.samples)

    def _make_mashup(self, genre_idx):
        genre = IDX2GENRE[genre_idx]
        songs = self.genre_songs[genre]
        n = random.randint(2, min(4, len(songs)))
        chosen = random.sample(songs, n)
        stems = []
        for s in chosen:
            st = random.choice(s['stems'])
            wav = load_wav(os.path.join(s['dir'], f"{st}.wav"))
            vol = random.uniform(0.3, 1.5) * STEM_WEIGHTS.get(st, 0.3) / 0.35
            stems.append(wav * vol)
        mix = torch.stack(stems).sum(dim=0)

        if self.augment:
            mix = torch.roll(mix, random.randint(-CFG.sr, CFG.sr))
            for _ in range(random.randint(0, 2)):
                noise = load_wav(random.choice(self.noise_files))
                snr = random.uniform(5.0, 25.0)
                sp = mix.pow(2).mean() + 1e-10
                np_ = noise.pow(2).mean() + 1e-10
                mix = mix + noise * (sp / (np_ * 10**(snr/10))).sqrt()
            if random.random() < 0.3:
                mix = torch.clamp(mix * random.uniform(1.2, 3.0), -1, 1)
            if random.random() < 0.3:
                shift = random.uniform(-2, 2)
                rate = 2 ** (shift / 12)
                mix = torchaudio.functional.resample(mix.unsqueeze(0), CFG.sr, int(CFG.sr * rate)).squeeze(0)
                if len(mix) < TARGET_LEN: mix = F.pad(mix, (0, TARGET_LEN - len(mix)))
                else: mix = mix[:TARGET_LEN]

        peak = mix.abs().max()
        if peak > 1e-6: mix = mix / peak * random.uniform(0.5, 1.0)
        return mix

    def __getitem__(self, idx):
        gi = self.samples[idx]
        mix = self._make_mashup(gi)

        if self.mixup_alpha > 0 and random.random() < 0.5:
            gi2 = random.choice(self.samples)
            mix2 = self._make_mashup(gi2)
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            mix = lam * mix + (1 - lam) * mix2
            label = torch.zeros(len(CFG.genres))
            label[gi] = lam; label[gi2] += (1 - lam)
            return mix, label
        return mix, gi


class ValDataset(Dataset):
    def __init__(self, song_list, label_list):
        self.items = list(zip(song_list, label_list))
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        song, label = self.items[idx]
        stems = []
        for st in song['stems']:
            stems.append(load_wav(os.path.join(song['dir'], f"{st}.wav")))
        mix = torch.stack(stems).sum(dim=0)
        peak = mix.abs().max()
        if peak > 1e-6: mix = mix / peak
        return mix, label


class TestDataset(Dataset):
    def __init__(self):
        self.df = pd.read_csv(CFG.test_csv, dtype={'id': str})
        self.paths = []
        for _, row in self.df.iterrows():
            path = None
            for pat in [f"song{str(row['id']).zfill(4)}.wav", f"{row['id']}.wav", f"song{row['id']}.wav"]:
                p = os.path.join(CFG.test_dir, pat)
                if os.path.exists(p): path = p; break
            self.paths.append(path)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        path = self.paths[idx]
        if path is None: return torch.zeros(TARGET_LEN), str(self.df.iloc[idx]['id'])
        return load_wav(path), str(self.df.iloc[idx]['id'])

print("Datasets ready.")

Datasets ready.


## Model: MelSpec → EfficientNet-B0 → GeM → Linear(10)
- Mel spectrogram on GPU (fast)
- InstanceNorm for volume invariance
- GeM pooling > avg pool
- SpecAugment: freq + time masking on spectrogram

In [14]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.tensor(p))
        self.eps = eps
    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=(-2, -1)).pow(1.0 / self.p)


class GenreClassifier(nn.Module):
    def __init__(self, backbone='efficientnet_b0', pretrained=True, use_specaug=False):
        super().__init__()
        self.mel_spec = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sr, n_fft=CFG.n_fft, hop_length=CFG.hop_length,
            n_mels=CFG.n_mels, f_min=CFG.fmin, f_max=CFG.fmax)
        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(top_db=80)
        self.inst_norm = nn.InstanceNorm2d(1)

        # SpecAugment
        self.use_specaug = use_specaug
        if use_specaug:
            self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=20)
            self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=50)

        self.backbone = timm.create_model(backbone, pretrained=pretrained,
                                           in_chans=1, num_classes=0, global_pool='')
        nf = self.backbone.num_features
        self.gem = GeM(p=3.0)
        self.norm = nn.LayerNorm(nf)
        self.drop = nn.Dropout(0.5)
        self.fc = nn.Linear(nf, len(CFG.genres))

    def forward(self, x):
        with torch.no_grad():
            spec = self.mel_spec(x)
            spec = self.amp_to_db(spec)
            spec = spec.unsqueeze(1)
            spec = self.inst_norm(spec)
        if self.use_specaug and self.training:
            spec = self.freq_mask(spec)
            spec = self.time_mask(spec)
        feat = self.backbone(spec)
        p = self.gem(feat)
        p = self.norm(p)
        p = self.drop(p)
        return self.fc(p)

# Quick test
model = GenreClassifier().to(DEVICE)
dummy = torch.randn(2, TARGET_LEN).to(DEVICE)
with torch.no_grad(): out = model(dummy)
print(f"Output: {out.shape}")
params = sum(p.numel() for p in model.parameters())
print(f"Params: {params/1e6:.1f}M")
del model, dummy, out; gc.collect(); torch.cuda.empty_cache()

Output: torch.Size([2, 10])
Params: 4.0M


## Training & Evaluation

In [15]:
def mixup_criterion(criterion, logits, labels):
    """Handle both hard labels (int) and soft labels (one-hot from mixup)."""
    if isinstance(labels, torch.Tensor) and labels.dim() == 2:
        # Soft labels from mixup
        log_probs = F.log_softmax(logits, dim=1)
        return -(labels * log_probs).sum(dim=1).mean()
    return criterion(logits, labels)


def train_one_epoch(model, loader, optimizer, scheduler, scaler, criterion):
    model.train()
    total_loss, n = 0, 0
    for wav, labels in tqdm(loader, desc="  Train", leave=False):
        wav = wav.to(DEVICE)
        if isinstance(labels, torch.Tensor) and labels.dim() == 2:
            labels = labels.to(DEVICE)  # soft labels
        else:
            labels = labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            logits = model(wav)
            loss = mixup_criterion(criterion, logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * wav.size(0)
        n += wav.size(0)
    scheduler.step()
    return total_loss / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    for wav, labels in loader:
        wav = wav.to(DEVICE)
        with autocast():
            logits = model(wav)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = np.mean(np.array(all_preds) == np.array(all_labels))
    return f1, acc, np.array(all_preds), np.array(all_labels), np.vstack(all_probs)


@torch.no_grad()
def predict_tta(model, loader, n_tta=5):
    model.eval()
    all_probs = []
    for t in range(n_tta):
        batch_probs = []
        for wav, ids in loader:
            wav = wav.to(DEVICE)
            with autocast():
                logits = model(wav)
            batch_probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs.append(np.vstack(batch_probs))
    avg = np.mean(all_probs, axis=0)
    return avg.argmax(axis=1), avg


def run_training(model, train_songs, train_labels, val_songs, val_labels,
                 noise_files, epochs=30, lr=1e-3, samples_per_genre=800,
                 batch_size=16, use_specaug=False, mixup_alpha=0.0, run_name=""):
    """Full training loop. Returns best_f1 and best model state."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = GradScaler()

    val_ds = ValDataset(val_songs, val_labels)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    best_f1, best_state = 0.0, None
    history = {'loss': [], 'f1': [], 'acc': []}

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_ds = MashupDataset(train_songs, train_labels, noise_files,
                                  samples_per_genre=samples_per_genre, augment=True,
                                  use_specaug=use_specaug, mixup_alpha=mixup_alpha)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                                   num_workers=2, pin_memory=True, drop_last=True)
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, criterion)
        f1, acc, _, _, _ = evaluate(model, val_loader)
        history['loss'].append(loss); history['f1'].append(f1); history['acc'].append(acc)

        tag = ""
        if f1 > best_f1:
            best_f1 = f1
            best_state = copy.deepcopy(model.state_dict())
            tag = " ★"

        wandb.log({f"{run_name}/loss": loss, f"{run_name}/f1": f1,
                   f"{run_name}/acc": acc, f"{run_name}/lr": scheduler.get_last_lr()[0]})
        print(f"  [{epoch:02d}/{epochs}] loss={loss:.4f} f1={f1:.4f} acc={acc:.4f} ({time.time()-t0:.0f}s){tag}")

    return best_f1, best_state, history

print("Training functions ready.")

Training functions ready.


## Run 3a — EfficientNet-B0 Baseline
No SpecAugment, no Mixup. Just mashup augmentation + noise.

In [ ]:
# 85/15 split
from sklearn.model_selection import train_test_split
tr_idx, val_idx = train_test_split(range(len(all_songs)), test_size=0.15,
                                    stratify=all_labels, random_state=SEED)
tr_songs = [all_songs[i] for i in tr_idx]; tr_labels = all_labels[tr_idx]
va_songs = [all_songs[i] for i in val_idx]; va_labels = all_labels[val_idx]
print(f"Train: {len(tr_songs)}, Val: {len(va_songs)}")

model_3a = GenreClassifier(backbone='efficientnet_b0', pretrained=True, use_specaug=False).to(DEVICE)
print("\n--- Run 3a: Baseline ---")
f1_3a, state_3a, hist_3a = run_training(
    model_3a, tr_songs, tr_labels, va_songs, va_labels, noise_files,
    epochs=25, lr=1e-3, samples_per_genre=800, batch_size=16,
    use_specaug=False, mixup_alpha=0.0, run_name="3a_baseline")
torch.save(state_3a, os.path.join(CFG.output_dir, 'best_3a_baseline.pth'))
print(f"\n3a Best F1: {f1_3a:.4f}")
wandb.log({"3a/best_f1": f1_3a})

Train: 850, Val: 150

--- Run 3a: Baseline ---


  Train:   0%|          | 0/500 [00:00<?, ?it/s]

## Run 3b — + SpecAugment + Mixup
Add frequency/time masking on spectrograms + Mixup (alpha=0.3).

In [ ]:
model_3b = GenreClassifier(backbone='efficientnet_b0', pretrained=True, use_specaug=True).to(DEVICE)
print("\n--- Run 3b: SpecAugment + Mixup ---")
f1_3b, state_3b, hist_3b = run_training(
    model_3b, tr_songs, tr_labels, va_songs, va_labels, noise_files,
    epochs=25, lr=1e-3, samples_per_genre=800, batch_size=16,
    use_specaug=True, mixup_alpha=0.3, run_name="3b_specaug_mixup")
torch.save(state_3b, os.path.join(CFG.output_dir, 'best_3b_specaug.pth'))
print(f"\n3b Best F1: {f1_3b:.4f}")
wandb.log({"3b/best_f1": f1_3b})

### Compare 3a vs 3b

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(hist_3a['loss'], label='3a baseline'); axes[0].plot(hist_3b['loss'], label='3b specaug+mixup')
axes[0].set_title('Training Loss'); axes[0].legend()
axes[1].plot(hist_3a['f1'], label=f"3a (best={f1_3a:.4f})"); axes[1].plot(hist_3b['f1'], label=f"3b (best={f1_3b:.4f})")
axes[1].set_title('Val F1'); axes[1].legend()
plt.suptitle('Run 3a vs 3b'); plt.tight_layout()
wandb.log({"plots/3a_vs_3b": wandb.Image(fig)}); plt.show()

# Pick better config
best_run = '3b' if f1_3b >= f1_3a else '3a'
best_specaug = best_run == '3b'
best_mixup = 0.3 if best_run == '3b' else 0.0
print(f"\nBetter config: Run {best_run} → use_specaug={best_specaug}, mixup={best_mixup}")

## Run 3c — 5-Fold Cross-Validation
Train 5 models on different splits. This gives:
1. Better F1 estimate (less variance)
2. 5 models for ensemble averaging
3. Out-of-fold predictions for stacking

In [ ]:
N_FOLDS = 5
FOLD_EPOCHS = 25
FOLD_SPG = 800  # samples per genre

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_models, fold_f1s = [], []
oof_preds = np.zeros((len(all_songs), len(CFG.genres)))  # out-of-fold probs

for fold, (tr_idx, va_idx) in enumerate(skf.split(all_songs, all_labels)):
    print(f"\n{'='*50}")
    print(f"FOLD {fold+1}/{N_FOLDS}")
    print(f"{'='*50}")

    f_tr_songs = [all_songs[i] for i in tr_idx]; f_tr_labels = all_labels[tr_idx]
    f_va_songs = [all_songs[i] for i in va_idx]; f_va_labels = all_labels[va_idx]

    model_fold = GenreClassifier(backbone='efficientnet_b0', pretrained=True,
                                  use_specaug=best_specaug).to(DEVICE)
    f1_fold, state_fold, _ = run_training(
        model_fold, f_tr_songs, f_tr_labels, f_va_songs, f_va_labels, noise_files,
        epochs=FOLD_EPOCHS, lr=1e-3, samples_per_genre=FOLD_SPG, batch_size=16,
        use_specaug=best_specaug, mixup_alpha=best_mixup, run_name=f"3c_fold{fold+1}")

    # Save fold model
    save_path = os.path.join(CFG.output_dir, f'best_fold{fold+1}.pth')
    torch.save(state_fold, save_path)
    fold_f1s.append(f1_fold)
    fold_models.append(save_path)

    # OOF predictions
    model_fold.load_state_dict(state_fold)
    val_ds = ValDataset(f_va_songs, f_va_labels)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    _, _, _, _, val_probs = evaluate(model_fold, val_loader)
    oof_preds[va_idx] = val_probs

    wandb.log({f"3c/fold{fold+1}_f1": f1_fold})
    print(f"  Fold {fold+1} best F1: {f1_fold:.4f}")

    del model_fold; gc.collect(); torch.cuda.empty_cache()

print(f"\n5-Fold Results: {[f'{f:.4f}' for f in fold_f1s]}")
print(f"Mean F1: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}")
wandb.log({"3c/mean_f1": np.mean(fold_f1s), "3c/std_f1": np.std(fold_f1s)})

# Save OOF
np.save(os.path.join(CFG.output_dir, 'oof_preds_efficientnet.npy'), oof_preds)

### OOF Confusion Matrix

In [ ]:
oof_pred_labels = oof_preds.argmax(axis=1)
oof_f1 = f1_score(all_labels, oof_pred_labels, average='macro')
print(f"OOF F1: {oof_f1:.4f}")
print(classification_report(all_labels, oof_pred_labels, target_names=CFG.genres))

fig, ax = plt.subplots(figsize=(10, 8))
cm = confusion_matrix(all_labels, oof_pred_labels)
ConfusionMatrixDisplay(cm, display_labels=CFG.genres).plot(ax=ax, cmap='Blues', xticks_rotation=45)
ax.set_title(f'OOF Confusion Matrix — F1={oof_f1:.4f}')
plt.tight_layout()
wandb.log({"plots/oof_confusion": wandb.Image(fig)}); plt.show()

---
## Run 3d — Pseudo-Labeling
1. Run inference on test set with fold-averaged predictions
2. Take high-confidence predictions (prob > 0.95)
3. Add them to training set
4. Retrain

In [ ]:
# Step 1: Get test predictions from all folds
test_ds = TestDataset()
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

test_probs_all = []
for fold_path in fold_models:
    model_pl = GenreClassifier(backbone='efficientnet_b0', pretrained=True,
                                use_specaug=best_specaug).to(DEVICE)
    model_pl.load_state_dict(torch.load(fold_path, map_location=DEVICE))
    _, probs = predict_tta(model_pl, test_loader, n_tta=3)
    test_probs_all.append(probs)
    del model_pl; gc.collect(); torch.cuda.empty_cache()

test_probs_avg = np.mean(test_probs_all, axis=0)
test_preds = test_probs_avg.argmax(axis=1)
test_confidence = test_probs_avg.max(axis=1)

print(f"Test predictions: {Counter(test_preds)}")
print(f"Confidence stats: mean={test_confidence.mean():.3f}, "
      f">0.90: {(test_confidence > 0.90).sum()}, "
      f">0.95: {(test_confidence > 0.95).sum()}")

In [ ]:
# Step 2: Create pseudo-labeled dataset
CONF_THRESHOLD = 0.95
pseudo_mask = test_confidence > CONF_THRESHOLD
pseudo_indices = np.where(pseudo_mask)[0]
print(f"Pseudo-labeled samples: {len(pseudo_indices)} / {len(test_ds)} "
      f"({100*len(pseudo_indices)/len(test_ds):.1f}%)")
print(f"Per genre: {Counter(test_preds[pseudo_indices])}")

class PseudoDataset(Dataset):
    """Wraps test samples as labeled training data."""
    def __init__(self, test_ds, indices, labels):
        self.test_ds = test_ds
        self.indices = indices
        self.labels = labels
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        wav, _ = self.test_ds[self.indices[idx]]
        return wav, self.labels[idx]

# Step 3: Retrain best fold with pseudo labels mixed in
if len(pseudo_indices) > 100:
    print(f"\nRetraining with pseudo-labels...")
    # Use fold 1 split
    tr_idx_f1 = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(all_songs, all_labels))[0][0]
    va_idx_f1 = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(all_songs, all_labels))[0][1]

    model_3d = GenreClassifier(backbone='efficientnet_b0', pretrained=True,
                                use_specaug=best_specaug).to(DEVICE)
    # Load best fold 1 weights as starting point
    model_3d.load_state_dict(torch.load(fold_models[0], map_location=DEVICE))

    f_tr_songs = [all_songs[i] for i in tr_idx_f1]; f_tr_labels = all_labels[tr_idx_f1]
    f_va_songs = [all_songs[i] for i in va_idx_f1]; f_va_labels = all_labels[va_idx_f1]

    # Shorter fine-tune with pseudo labels
    f1_3d, state_3d, hist_3d = run_training(
        model_3d, f_tr_songs, f_tr_labels, f_va_songs, f_va_labels, noise_files,
        epochs=10, lr=3e-4, samples_per_genre=600, batch_size=16,
        use_specaug=best_specaug, mixup_alpha=best_mixup, run_name="3d_pseudo")
    torch.save(state_3d, os.path.join(CFG.output_dir, 'best_3d_pseudo.pth'))
    print(f"3d F1 (with pseudo): {f1_3d:.4f}")
    wandb.log({"3d/best_f1": f1_3d})
else:
    print("Not enough confident pseudo-labels, skipping.")
    f1_3d = 0.0

## Final Inference — 5-Fold Average + TTA
Average probabilities from all 5 folds × 5 TTA runs = 25 forward passes per sample.

In [ ]:
test_ds = TestDataset()
test_loader = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

final_probs = []
for fold_path in tqdm(fold_models, desc="Fold inference"):
    model_inf = GenreClassifier(backbone='efficientnet_b0', pretrained=True,
                                 use_specaug=False).to(DEVICE)  # no specaug at inference
    model_inf.load_state_dict(torch.load(fold_path, map_location=DEVICE))
    _, probs = predict_tta(model_inf, test_loader, n_tta=5)
    final_probs.append(probs)
    del model_inf; gc.collect(); torch.cuda.empty_cache()

final_avg = np.mean(final_probs, axis=0)
final_preds = final_avg.argmax(axis=1)
print(f"Final predictions: {Counter(final_preds)}")

## Submission

In [ ]:
test_df = pd.read_csv(CFG.test_csv, dtype={'id': str})
test_df['genre'] = [IDX2GENRE[p] for p in final_preds]
sub_path = os.path.join(CFG.output_dir, 'submission_cnn.csv')
test_df.to_csv(sub_path, index=False)
print(f"Saved: {sub_path}")
print(test_df['genre'].value_counts().sort_index())
print(test_df.head(10))

# Save probs for ensemble
np.save(os.path.join(CFG.output_dir, 'test_probs_efficientnet.npy'), final_avg)
np.save(os.path.join(CFG.output_dir, 'oof_preds_efficientnet.npy'), oof_preds)

## Results Summary

In [ ]:
results = {
    "3a_baseline": f1_3a,
    "3b_specaug_mixup": f1_3b,
    "3c_5fold_mean": np.mean(fold_f1s),
    "3c_oof": oof_f1,
}
if f1_3d > 0: results["3d_pseudo"] = f1_3d

fig, ax = plt.subplots(figsize=(10, 5))
names, scores = zip(*results.items())
colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(scores)))
ax.barh(names, scores, color=colors)
for i, v in enumerate(scores): ax.text(v + 0.003, i, f'{v:.4f}', va='center')
ax.set_xlabel('Macro F1'); ax.set_title('CNN Iteration Results')
ax.set_xlim(0, 1); plt.tight_layout()
wandb.log({"plots/cnn_results": wandb.Image(fig)}); plt.show()

print("\n" + "="*50)
print("CNN PHASE RESULTS")
print("="*50)
for k, v in results.items(): print(f"  {k}: {v:.4f}")
print(f"\nFiles saved for ensemble:")
print(f"  test_probs_efficientnet.npy")
print(f"  oof_preds_efficientnet.npy")
print(f"  5 fold checkpoints")

In [ ]:
wandb.log({"cnn/best_single_f1": max(f1_3a, f1_3b),
           "cnn/5fold_mean_f1": np.mean(fold_f1s), "cnn/oof_f1": oof_f1})
art = wandb.Artifact("cnn_efficientnet_folds", type="model")
for fp in fold_models: art.add_file(fp)
run.log_artifact(art)
wandb.finish()
print("\n✅ exp_003 CNN complete.")